<a href="https://colab.research.google.com/github/tuckerlucy1/HLS-Data-Resources/blob/main/Gmas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# 🦾 AURORA PRIME | SYSTEM VERSION: GOLDEN MASTER (BI-DIRECTIONAL)
# ==============================================================================
# INTEGRATED SYSTEMS:
# 1. ARC REACTOR (Physics) - Detects CRASHES (-Gamma) and ROCKETS (+Gamma)
# 2. J.A.R.V.I.S. (Voice) - 6:00 AM Wake-up Briefing
# 3. SAFETY SUITE - Hard Deck & Mid-Day Lockout (Earnings Lock REMOVED)
# 4. FRIDAY GOVERNOR - Automatically tightens entry rules on Fridays
# ==============================================================================

# --- 0. SYSTEM INSTALLATION (The Spark Plug) ---
!pip install gtts -q
!pip install yfinance -q

import time
import pandas as pd
import yfinance as yf
import numpy as np
from datetime import datetime
import pytz
from IPython.display import clear_output, display, Audio
from gtts import gTTS

# --- 1. CONFIGURATION: THE ARMORY ---
# ⚠️ EARNINGS LOCK REMOVED: Bot will trade AVGO/ORCL regardless of news.
ROSTER = ["MSTR", "COIN", "ORCL", "NVDA", "QQQ", "SQQQ", "AVGO", "TQQQ"]
EARNINGS_RISK = []

# [OPTIONAL] BROKER KEYS (For Future Use - Leave blank for HUD Mode)
API_KEY = "PASTE_ALPACA_KEY_HERE"
SECRET_KEY = "PASTE_ALPACA_SECRET_HERE"

# --- 2. THE FRIDAY GOVERNOR ---
def get_rules():
    now = datetime.now(pytz.timezone('US/Pacific'))
    is_friday = (now.weekday() == 4)

    if is_friday:
        return {
            "MODE": "🚫 FRIDAY (STRICT)",
            "GAMMA_TRIGGER": 0.80,   # Needs +/- 0.80 speed to fire (Very Fast)
            "STOP_HOUR": 9,          # Stop trading at 9:30 AM
            "STOP_MINUTE": 30,
            "HARD_DECK": 5.0         # Stricter stop limits (5%)
        }
    else:
        return {
            "MODE": "✅ STANDARD (AGGRESSIVE)",
            "GAMMA_TRIGGER": 0.70,   # Needs +/- 0.70 speed to fire
            "STOP_HOUR": 10,         # Stop trading at 10:00 AM
            "STOP_MINUTE": 0,
            "HARD_DECK": 8.0         # Standard stop limits (8%)
        }

# --- 3. J.A.R.V.I.S. VOICE MODULE ---
def jarvis_speak(text):
    try:
        tts = gTTS(text, lang='en', tld='us')
        tts.save('jarvis_alert.mp3')
        display(Audio('jarvis_alert.mp3', autoplay=True))
    except:
        pass

def run_wakeup_protocol():
    # Runs once at 6:00 AM to wake you up
    now = datetime.now(pytz.timezone('US/Pacific'))
    if now.hour == 6 and now.minute < 5:
        rules = get_rules()
        msg = f"Good morning sir. Aurora Prime is online. {rules['MODE']} protocols engaged. Scanning for pre-market volatility."
        jarvis_speak(msg)
        time.sleep(10)

# --- 4. THE ARC REACTOR (PHYSICS ENGINE) ---
def get_telemetry():
    rules = get_rules()
    trigger_val = rules["GAMMA_TRIGGER"]

    try:
        # Download last 5 days of 5m data for smooth math
        data = yf.download(ROSTER, period="5d", interval="5m", progress=False)
        hud_data = []

        for ticker in ROSTER:
            try:
                # Data Extraction
                if len(ROSTER) > 1: prices = data['Close'][ticker].dropna()
                else: prices = data['Close'].dropna()

                if len(prices) < 20: continue

                # A. PHYSICS (Gamma Calculation)
                velocity = prices.diff()
                acceleration = velocity.diff()
                gamma = -1 * velocity.rolling(14).corr(acceleration).iloc[-1]

                # B. HARD DECK (Day Change)
                current_price = prices.iloc[-1]
                open_price = prices.iloc[-20] # Approx 2 hours ago
                day_change = ((current_price - open_price) / open_price) * 100

                # C. DECISION MATRIX (FULL SPECTRUM)
                status = "💤 SCANNING"
                color = "⚪"

                # Safety Checks
                if abs(day_change) > rules["HARD_DECK"] and ticker not in ["SQQQ", "TQQQ"]:
                     # If stock moved > 8% already, be careful (unless it's an ETF)
                    status = "🛡️ EXTENDED (WAIT)"
                    color = "🔵"

                # ATTACK LOGIC
                else:
                    # 1. THE CRASH (PUTS)
                    if gamma < (trigger_val * -1): # e.g. Gamma < -0.70
                        status = "🚨 CRASH (BUY PUTS)"
                        color = "🔴"

                    # 2. THE ROCKET (CALLS)
                    elif gamma > trigger_val: # e.g. Gamma > +0.70
                        status = "🚀 ROCKET (BUY CALLS)"
                        color = "🟢"

                    # 3. MOMENTUM BUILDING
                    elif gamma < -0.50:
                        status = "⚠️ SLIDING..."
                        color = "🟠"
                    elif gamma > 0.50:
                        status = "🔋 CLIMBING..."
                        color = "🟢"

                hud_data.append({
                    "ticker": ticker,
                    "price": current_price,
                    "gamma": gamma,
                    "change": day_change,
                    "status": status,
                    "color": color
                })

            except: continue

        # Sort by Intensity (Absolute Gamma)
        hud_data.sort(key=lambda x: abs(x['gamma']), reverse=True)
        return hud_data

    except:
        return []

# --- 5. MAIN LOOP ---
print("🦾 AURORA PRIME INITIALIZING...")
print("⚠️ WARNING: EARNINGS SAFETY OFF. BI-DIRECTIONAL MODE ACTIVE.")
run_wakeup_protocol()

try:
    while True:
        clear_output(wait=True)
        now = datetime.now(pytz.timezone('US/Pacific'))
        rules = get_rules()
        time_str = now.strftime('%H:%M:%S')

        # TIME LOCK (Friday = 9:30 AM Stop / Normal = 10:00 AM Stop)
        is_trading_hours = True
        if now.hour > rules["STOP_HOUR"]: is_trading_hours = False
        elif now.hour == rules["STOP_HOUR"] and now.minute >= rules["STOP_MINUTE"]: is_trading_hours = False
        if now.hour < 6: is_trading_hours = False # Pre-market wait

        # --- THE HUD ---
        print(f"░▒▓█ AURORA PRIME HUD █▓▒░      TIME: {time_str}")
        print(f"   >> PROTOCOL: {rules['MODE']}")
        print(f"   >> TRIGGER:  +/- {rules['GAMMA_TRIGGER']}")
        print("━" * 75)

        if not is_trading_hours:
            print(f"\n🛑 TRADING LOCKOUT ACTIVE (Session Closed)")
            print(f">> REASON: Outside safe operating hours ({rules['STOP_HOUR']}:{rules['STOP_MINUTE']} limit).")
            print(">> ACTION: Stand down. Enjoy the day.")
            time.sleep(60)
            continue

        # LIVE TELEMETRY
        telemetry = get_telemetry()
        print(f"{'  ASSET':<8} | {'PRICE':<9} | {'GAMMA':<8} | {'CHANGE':<8} | {'STATUS'}")
        print("━" * 75)

        active_target = False
        target_type = ""

        for row in telemetry:
            print(f"{row['color']} {row['ticker']:<6} | ${row['price']:<8.2f} | {row['gamma']:>.4f}   | {row['change']:>.2f}%    | {row['status']}")
            if "BUY" in row['status']:
                active_target = True
                target_type = "CALLS" if "ROCKET" in row['status'] else "PUTS"

        print("━" * 75)

        # TACTICAL ADVICE
        if active_target:
            print(f"\n🎯 TACTICAL ALERT: TARGET LOCKED ({target_type}).")
            print(">> EXECUTE: Check Option Spreads. Fire immediately.")
        else:
            print("\n💤 MARKET STATUS: STABLE.")
            print(">> ACTION: Wait for 'Morning Drive' volatility.")

        time.sleep(30)

except KeyboardInterrupt:
    print("SYSTEM SHUTDOWN.")


░▒▓█ AURORA PRIME HUD █▓▒░      TIME: 20:49:26
   >> PROTOCOL: ✅ STANDARD (AGGRESSIVE)
   >> TRIGGER:  +/- 0.7
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🛑 TRADING LOCKOUT ACTIVE (Session Closed)
>> REASON: Outside safe operating hours (10:0 limit).
>> ACTION: Stand down. Enjoy the day.
